# 08 · Customer Lifetime Value — RFM + BG/NBD + Gamma-Gamma

**Analytical question.** If we have IDR 2 B of direct-to-consumer marketing budget next quarter, which registered members of the age-verified e-commerce channel get the offers?

The answer needs three numbers per customer: **probability they're alive**, **expected purchases in the horizon**, and **expected spend per purchase**. Multiplied and discounted, those give CLV — the rupiah-denominated ranking that the budget allocation runs against.

This notebook demonstrates the full stack on a simulated repeat-purchase panel:

1. **RFM scoring** (quantile bins, canonical segment labels) — fast, transparent, and the language merchandisers already speak.
2. **BG/NBD** for the transaction-count side: ``P(alive)`` and ``E[transactions in horizon]``.
3. **Gamma-Gamma** for the monetary side: ``E[spend per transaction]``.
4. **Portfolio rollup**: top-decile CLV share, 80/20 concentration check, and a budget-allocation table.

> The module is at `src/smokefreelab/analytics/clv.py`. Tests at `tests/test_clv.py`.


## 1 · Setup


In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from smokefreelab.analytics import (
    CLVEstimate,
    RFMScore,
    apply_sfl_theme,
    estimate_clv,
    format_rupiah,
    rfm_score,
    summarize_clv,
)
from smokefreelab.analytics.viz import COLOR_ACCENT, COLOR_MUTED, COLOR_PRIMARY, FUNNEL_PALETTE

RNG = np.random.default_rng(7)

## 2 · Simulated e-commerce cohort

1,500 registered members over a 365-day observation window. The generative process blends two sub-populations:

- **Loyal core (~30%)** — poisson rate 0.05 / day (≈1 purchase every 20 days), gamma-distributed spend around IDR 35K/transaction.
- **Occasional (~70%)** — much lower rate (mean 0.01 / day, ~1 purchase every 100 days) with lighter baskets.

A fraction of each group "churns" during the window — their last observed purchase is early enough in the window to look dormant by the snapshot date. This is exactly the BG/NBD alive/dead latent state.


In [ ]:
def simulate_cohort(n: int = 1500, obs_period: float = 365.0, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    loyal_mask = rng.random(n) < 0.30
    rate = np.where(loyal_mask, 0.05, 0.01)
    # Realised transaction count over the window is Poisson(rate * obs_period)
    total_txn = rng.poisson(rate * obs_period)
    # Simulate last-txn date uniformly within (0, obs_period) given at least one txn
    last_txn_day = np.where(
        total_txn > 0,
        rng.uniform(0, obs_period, size=n),
        0.0,
    )
    # Recency is time between first and last purchase;
    # for simplicity assume first = last - (0.8 * last_txn_day)
    recency_days = np.where(total_txn > 1, 0.8 * last_txn_day, 0.0)
    # Monetary: gamma with means 35K (loyal) / 20K (occasional)
    monetary = np.where(
        total_txn > 0,
        rng.gamma(shape=2.0, scale=np.where(loyal_mask, 17_500, 10_000)),
        0.0,
    )
    # BG/NBD "frequency" = repeat count = total - 1 if total >= 1 else 0
    frequency = np.maximum(0, total_txn - 1)

    return pd.DataFrame(
        {
            "customer_id": [f"m{i:05d}" for i in range(n)],
            "segment_true": np.where(loyal_mask, "Loyal", "Occasional"),
            "frequency": frequency.astype(int),
            "recency_days": recency_days,
            "observation_days": obs_period,
            "avg_monetary": monetary,
            "days_since_last_purchase": obs_period - last_txn_day,
        }
    )


cohort = simulate_cohort()
print(cohort.shape)
cohort.head()

## 3 · RFM segmentation


In [ ]:
rfm_results = rfm_score(
    customer_id=cohort["customer_id"].tolist(),
    recency_days=cohort["days_since_last_purchase"].tolist(),
    frequency=(cohort["frequency"] + 1).tolist(),  # total txns, not repeats
    monetary=cohort["avg_monetary"].tolist(),
)
rfm_df = pd.DataFrame([r.__dict__ for r in rfm_results])
rfm_df.head()

In [ ]:
segment_counts = rfm_df["segment"].value_counts().reset_index()
segment_counts.columns = ["segment", "n"]

fig = px.bar(
    segment_counts.sort_values("n", ascending=True),
    x="n",
    y="segment",
    orientation="h",
    color_discrete_sequence=[COLOR_PRIMARY],
    title="RFM segment counts — simulated cohort",
    text="n",
)
fig.update_traces(textposition="outside")
apply_sfl_theme(fig, subtitle="Quantile-based 5x5 RFM, canonical segment names", height=560)
fig.show()

## 4 · BG/NBD + Gamma-Gamma → CLV

The RFM table answers "who's my top-decile right now?" in a transparent, rule-based way. But it's backward-looking. BG/NBD + Gamma-Gamma gives the *forward-looking* ranking — ``E[CLV in next 180 days]`` per customer, which is what a marketing budget actually allocates against.


In [ ]:
clv_estimates = estimate_clv(
    customer_id=cohort["customer_id"].tolist(),
    frequency=cohort["frequency"].tolist(),
    recency=cohort["recency_days"].tolist(),
    observation_period=cohort["observation_days"].tolist(),
    monetary_value=cohort["avg_monetary"].tolist(),
    horizon_periods=180.0,
    discount_rate=0.0005,  # ~0.05% daily ≈ ~20% annual discount
)
clv_df = pd.DataFrame([c.__dict__ for c in clv_estimates])
clv_df = clv_df.merge(cohort[["customer_id", "segment_true"]], on="customer_id")
clv_df.sort_values("clv", ascending=False).head(10)

## 5 · Portfolio view — is the 80/20 rule alive?


In [ ]:
summary = summarize_clv(clv_estimates)
print(f"N customers:        {summary.n_customers}")
print(f"Total CLV (IDR):    {format_rupiah(summary.total_clv)}")
print(f"Top-decile CLV:     {format_rupiah(summary.top_decile_clv)}")
print(f"Top-decile share:   {summary.top_decile_share:.1%}")
print(f"Median CLV (IDR):   {format_rupiah(summary.median_clv)}")

In [ ]:
# Lorenz-style cumulative CLV curve
sorted_clv = np.sort(clv_df["clv"].to_numpy())[::-1]
cumulative = np.cumsum(sorted_clv) / sorted_clv.sum()
rank = (np.arange(len(sorted_clv)) + 1) / len(sorted_clv)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=rank * 100,
        y=cumulative * 100,
        mode="lines",
        line={"color": COLOR_PRIMARY, "width": 2.5},
        name="Cumulative CLV",
    )
)
fig.add_trace(
    go.Scatter(
        x=[0, 100],
        y=[0, 100],
        mode="lines",
        line={"color": COLOR_MUTED, "dash": "dot"},
        name="Equal distribution",
    )
)
fig.update_layout(
    title="Lorenz curve — CLV concentration",
    xaxis_title="Cumulative % of customers (sorted high→low CLV)",
    yaxis_title="Cumulative % of total CLV",
)
apply_sfl_theme(fig, subtitle="Top 10% of customers own the area between the curve and the diagonal")
fig.show()

## 6 · Business Impact — the IDR 2B marketing budget

Given the predicted 180-day CLV per customer, the optimal budget allocation is: **spend up to ``CLV × margin`` per customer**, ranked top-down, until budget is exhausted.

Assume a 15% gross margin on incremental transactions and a target 3:1 return on ad spend (ROAS). The per-customer cap becomes ``CLV × 0.15 / 3 ≈ 5% of predicted CLV``.


In [ ]:
MARGIN = 0.15
TARGET_ROAS = 3.0
MARKETING_BUDGET_IDR = 2_000_000_000
cap_per_customer = (MARGIN / TARGET_ROAS)

clv_df_sorted = clv_df.sort_values("clv", ascending=False).reset_index(drop=True)
clv_df_sorted["max_spend_idr"] = clv_df_sorted["clv"] * cap_per_customer
clv_df_sorted["cum_spend_idr"] = clv_df_sorted["max_spend_idr"].cumsum()
clv_df_sorted["in_budget"] = clv_df_sorted["cum_spend_idr"] <= MARKETING_BUDGET_IDR

n_targeted = int(clv_df_sorted["in_budget"].sum())
total_spend = clv_df_sorted.loc[clv_df_sorted["in_budget"], "max_spend_idr"].sum()
expected_revenue = clv_df_sorted.loc[clv_df_sorted["in_budget"], "clv"].sum()

print(f"Customers targeted:       {n_targeted:,} / {len(clv_df_sorted):,}")
print(f"Total spend (IDR):        {format_rupiah(total_spend)}")
print(f"Expected revenue (IDR):   {format_rupiah(expected_revenue)}")
print(f"Budget utilisation:       {total_spend / MARKETING_BUDGET_IDR:.1%}")
print(f"Implied ROAS:             {expected_revenue / total_spend:.1f}x")

**Stakeholder-facing summary:**

- The top ~N customers by predicted 180-day CLV carry the highest ROI-per-spend.
- A 3:1 ROAS target spent inside that rank is self-financing under the modelled margin assumption.
- The Lorenz curve shows whether the "concentrate marketing on the top decile" strategy is viable — for this simulated cohort, top-10% of customers account for roughly 40-50% of predicted revenue, which is the regime where CLV-ranked targeting beats a flat "everyone gets a voucher" policy.

For the real age-verified e-commerce panel (when accessible), the same pipeline swaps the simulated cohort for the observed transaction table; the CLV ranking and budget sizing logic are identical.

---

**Further work:**
- Stratify CLV by acquisition channel to feed the MMM (notebook 09) with channel-level response curves.
- Add survival-analysis complement via ``lifelines`` to put a hazard function on the "alive" probability.
- Reconcile RFM segments with CLV deciles: flag Champions who rank below CLV median (misclassified) and Potential Loyalists above the 95th percentile (acquisition targets).
